In [1]:
import torch

# check that model can answer correctly

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL)

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map={"": "cuda:0"})
# bf 16 so that we can fit in memory
# device map so that the weights are copied in GPU

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [3]:
prompt = "Answer with only one number. The number of legs on the animal that spins webs is?"

messages = [
    { "role": "user",
      "content": (prompt),
    }
]

chat_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)
    


input_tokens = tokenizer(chat_text, return_tensors='pt').to(model.device)
print(f'input tokens: {input_tokens}')
with torch.inference_mode():
    output_tokens = model.generate(**input_tokens)

print(f'output tokens: {output_tokens}')

input tokens: {'input_ids': tensor([[151644,    872,    198,  16141,    448,   1172,    825,   1372,     13,
            576,   1372,    315,  14201,    389,    279,   9864,    429,  44758,
          80920,    374,     30, 151645,    198, 151644,  77091,    198, 151667,
            271, 151668,    271]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]], device='cuda:0')}


/workspace/.venv/lib/python3.11/site-packages/transformers/generation/utils.py:1638: UserWarning: Using the model-agnostic default `max_length` (=50) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


output tokens: tensor([[151644,    872,    198,  16141,    448,   1172,    825,   1372,     13,
            576,   1372,    315,  14201,    389,    279,   9864,    429,  44758,
          80920,    374,     30, 151645,    198, 151644,  77091,    198, 151667,
            271, 151668,    271,     23, 151645]], device='cuda:0')


In [4]:
last_input_token_id = input_tokens['input_ids'].shape[1]
# the prediction will be from the last token
from_last_token_predictions = output_tokens[0, last_input_token_id:]
print(f'from last token predictions: {from_last_token_predictions}')

from last token predictions: tensor([    23, 151645], device='cuda:0')


In [5]:
decoded_response = tokenizer.decode(from_last_token_predictions, skip_special_tokens=True)
print(f'decoded_response: {decoded_response}')

decoded_response: 8


In [6]:
# j lens

In [7]:
import jlens

jlens_model = jlens.from_hf(model, tokenizer, force_bos=False)

In [8]:
print(jlens_model)

HFLensModel(Qwen3ForCausalLM, n_layers=36, d_model=4096)


In [13]:
jlens_model.layers[-2]

Qwen3DecoderLayer(
  (self_attn): Qwen3Attention(
    (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
    (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
    (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
    (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
  )
  (mlp): Qwen3MLP(
    (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
    (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
  (post_attention_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
)

In [15]:
from jlens.examples import load_wikitext_prompts

prompts = load_wikitext_prompts(n_prompts=10)

lens = jlens.fit(jlens_model, prompts, source_layers=[32,33,34])

JacobianLens(d_model=4096, n_prompts=10, source_layers=[32..34] (3 layers))